# 01 — Leakage-free benchmark of 24 configurations (Smoke Free)

12 model families × 2 feature sets (original 33, expanded 46), two tasks (same-entry classification, next-entry prediction).
Participant-level 10-fold cross-validation (KFold, seed 42) with nested tuning: 20% of each training fold's participants
form a validation set; standardisation, tuning, early stopping and self-supervised pretraining use training participants only.

**Produces:** Table 6 (AUC with 95% t-intervals), Table 7 (expanded vs original, Wilcoxon + Holm), per-entry test predictions
(`SAVE_DIR/predictions/`), per-participant AUC (Supplement S4), and the self-supervised early-stopping check (Supplement S3, Cells 13–15).

**Run:** put `smokefree_data.xlsx` next to the notebook (see `data/README.md`), set `QUICK_TEST = True` for a 2-fold smoke test,
then `QUICK_TEST = False` and *Run all*. A GPU is recommended for the deep models. Finished folds are skipped after a restart.

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ════════════════════════════════════════════════════════
import torch, torch.nn as nn
import numpy as np, pandas as pd
import copy, json, os, time, io, random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (pack_padded_sequence,
                                  pad_packed_sequence)
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    print("XGBoost not found — installing...")
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip',
                    'install','xgboost','--quiet'])
    import xgboost as xgb
    HAS_XGB = True

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── RERUN SETTINGS: edit here ──────────────────────────
QUICK_TEST = False     # True = 2 folds, 1 epoch, 60 participants (checks the code runs)
GRUD_IMPL  = 'che2018' # 'che2018' = GRU-D with real time decay; 'legacy' = old code
SEED       = 42
RUN_ONLY   = None      # None = all 24 configs, or a list e.g. ['TFM_orig','TFM_exp']
TUNE       = 'all'     # 'all' | 'traditional' (RF/XGB/LR/SVM only) | 'none'
INNER_VAL_FRAC = 0.2   # share of training participants used for tuning
PATIENCE   = 5         # early-stopping patience (epochs) for deep models

SAVE_DIR = 'obj1_v2_results'
# Colab: keep results if the runtime disconnects
# from google.colab import drive; drive.mount('/content/drive')
# SAVE_DIR = '/content/drive/MyDrive/obj1_v2_results'

N_FOLDS    = 2 if QUICK_TEST else 10
FT_EPOCHS  = 1 if QUICK_TEST else 30
SSL_EPOCHS = 1 if QUICK_TEST else 30
if QUICK_TEST:
    SAVE_DIR = SAVE_DIR + '_quicktest'
os.makedirs(os.path.join(SAVE_DIR, 'predictions'), exist_ok=True)
print(f"QUICK_TEST={QUICK_TEST}  GRUD_IMPL={GRUD_IMPL}  results -> {SAVE_DIR}")
print("✅ Cell 1 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — LOCAL DATA LOADING + FEATURE ENGINEERING
# Original features : 33
# Expanded features : 33 + temporal lags + rolling stats
#                     + cyclical time = ~48
# ════════════════════════════════════════════════════════

from pathlib import Path
import time

import numpy as np
import pandas as pd


# ── Local Excel file path ──────────────────────────────
# If the Excel file is in the same folder as the notebook:
XLSX_PATH = Path("smokefree_data.xlsx")
if not XLSX_PATH.exists() and Path("../data/smokefree_data.xlsx").exists():
    XLSX_PATH = Path("../data/smokefree_data.xlsx")     # repository data/ folder
if not XLSX_PATH.exists() and Path("/content/smokefree_data.xlsx").exists():
    XLSX_PATH = Path("/content/smokefree_data.xlsx")   # Colab upload location

# If the file is somewhere else, replace the previous line
# with its complete Windows path, for example:


# ── Verify that the file exists ────────────────────────
if not XLSX_PATH.exists():
    raise FileNotFoundError(
        f"\nCould not find the Excel file:\n"
        f"{XLSX_PATH.resolve()}\n\n"
        "Place smokefree_data.xlsx in the same folder as "
        "this notebook, or set XLSX_PATH to its complete path."
    )

print(f"Loading: {XLSX_PATH.resolve()}")


# ── Read Excel file ────────────────────────────────────
try:
    df_raw = pd.read_excel(
        XLSX_PATH,
        sheet_name="All_Data",
        engine="openpyxl"
    )
except ValueError:
    print(
        "Sheet 'All_Data' was not found. "
        "Loading the first sheet instead."
    )
    df_raw = pd.read_excel(
        XLSX_PATH,
        sheet_name=0,
        engine="openpyxl"
    )

print(f"Raw shape: {df_raw.shape}")


# ── Check required columns ─────────────────────────────
REQUIRED_COLUMNS = [
    "account_id",
    "adjusted_craving_record_created",
    "craving_did_smoke",
    "craving_severity",
    "time_to_first_cig",
    "days_since_qd"
]

missing_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in df_raw.columns
]

if missing_columns:
    raise KeyError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ── Basic preprocessing ────────────────────────────────
df = df_raw.copy()

df["adjusted_craving_record_created"] = pd.to_datetime(
    df["adjusted_craving_record_created"],
    errors="coerce"
)

invalid_dates = df[
    "adjusted_craving_record_created"
].isna().sum()

if invalid_dates > 0:
    print(
        f"Warning: removing {invalid_dates} entries "
        "with invalid timestamps."
    )
    df = df.dropna(
        subset=["adjusted_craving_record_created"]
    ).copy()

df["craving_did_smoke"] = pd.to_numeric(
    df["craving_did_smoke"],
    errors="coerce"
).fillna(0)

df["craving_severity"] = pd.to_numeric(
    df["craving_severity"],
    errors="coerce"
).fillna(0)

df["days_since_qd"] = pd.to_numeric(
    df["days_since_qd"],
    errors="coerce"
).fillna(0)

df = (
    df.sort_values(
        [
            "account_id",
            "adjusted_craving_record_created"
        ]
    )
    .reset_index(drop=True)
)

df["label"] = (
    df["craving_did_smoke"]
    .astype(float)
    .clip(lower=0, upper=1)
)


# ── Keep patients with at least 20 entries ─────────────
patient_counts = df.groupby("account_id").size()

eligible_patients = patient_counts[
    patient_counts >= 20
].index

df = (
    df[df["account_id"].isin(eligible_patients)]
    .copy()
    .reset_index(drop=True)
)

if QUICK_TEST:
    _keep = sorted(df["account_id"].unique())[:60]
    df = df[df["account_id"].isin(_keep)].copy().reset_index(drop=True)

print(
    f"After minimum-entry filtering: "
    f"{df['account_id'].nunique()} patients, "
    f"{len(df):,} entries"
)


# Recreate groupby after filtering and sorting
grp = df.groupby("account_id", sort=False)


# ── Original features ──────────────────────────────────
ttfc_map = {
    "<= 5": 0,
    "6-30": 1,
    "31-60": 2,
    "> 60": 3
}

df["time_to_first_cig_ord"] = (
    df["time_to_first_cig"]
    .map(ttfc_map)
    .fillna(0)
    .astype(float)
)

df["prior_event_lapse"] = (
    grp["label"]
    .shift(1)
    .fillna(0)
    .astype(float)
)

df["mins_since_prev_rep"] = (
    grp["adjusted_craving_record_created"]
    .diff()
    .dt.total_seconds()
    .div(60)
    .fillna(0)
    .clip(lower=0)
)


FEELINGS = [
    "craving_severity",
    "craving_feeling_annoyed",
    "craving_feeling_anxious",
    "craving_feeling_bored",
    "craving_feeling_down",
    "craving_feeling_happy",
    "craving_feeling_hungry",
    "craving_feeling_lonely"
]

ACTIVITIES = [
    "craving_occupation_at_bar_or_event",
    "craving_occupation_chatting",
    "craving_occupation_coffee_or_tea",
    "craving_occupation_drinking",
    "craving_occupation_driving",
    "craving_occupation_going_to_bed",
    "craving_occupation_just_eaten",
    "craving_occupation_just_had_sex",
    "craving_occupation_reading",
    "craving_occupation_relaxing",
    "craving_occupation_taking_a_break",
    "craving_occupation_thinking",
    "craving_occupation_waking_up",
    "craving_occupation_working"
]

SOCIAL = [
    "craving_env_alone",
    "craving_env_colleagues",
    "craving_env_family",
    "craving_env_friends",
    "craving_env_partner",
    "craving_env_strangers"
]

TEMPORAL = [
    "cigs_per_day",
    "time_to_first_cig_ord",
    "days_since_qd",
    "mins_since_prev_rep",
    "prior_event_lapse"
]

ORIG_FEATS = [
    column
    for column in FEELINGS + ACTIVITIES + SOCIAL + TEMPORAL
    if column in df.columns
]


# Convert original feature columns to numeric
for column in ORIG_FEATS:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ── Expanded temporal features ─────────────────────────
hour = df[
    "adjusted_craving_record_created"
].dt.hour.astype(float)

df["hour_sin"] = np.sin(
    2 * np.pi * hour / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * hour / 24
)

df["is_weekend"] = (
    df["adjusted_craving_record_created"]
    .dt.dayofweek
    .ge(5)
    .astype(float)
)


# Recreate groupby after adding and converting columns
grp = df.groupby("account_id", sort=False)

df["severity_change"] = (
    grp["craving_severity"]
    .diff()
    .fillna(0)
)

df["rolling3_sev_mean"] = (
    grp["craving_severity"]
    .transform(
        lambda values: (
            values.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )
    .fillna(0)
)

df["rolling3_sev_std"] = (
    grp["craving_severity"]
    .transform(
        lambda values: (
            values.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .std()
        )
    )
    .fillna(0)
)


# ── Days since last previous lapse ─────────────────────
# unit-safe: seconds since 1970 whatever the timestamp precision (pandas 2 or 3)
EPOCH = pd.Timestamp("1970-01-01")
timestamp_seconds = (
    df["adjusted_craving_record_created"] - EPOCH
).dt.total_seconds()

df["_lapse_time"] = np.where(
    df["label"].eq(1),
    timestamp_seconds,
    np.nan
)

# Recreate groupby because _lapse_time was just added
grp = df.groupby("account_id", sort=False)

df["_last_lapse_time"] = (
    grp["_lapse_time"]
    .transform(
        lambda values: values.shift(1).ffill()
    )
)

patient_start_seconds = (
    grp["adjusted_craving_record_created"].transform("min") - EPOCH
).dt.total_seconds()

reference_time = df[
    "_last_lapse_time"
].fillna(patient_start_seconds)

df["days_since_last_lapse"] = (
    (timestamp_seconds - reference_time) / 86400
).clip(lower=0)

df.drop(
    columns=[
        "_lapse_time",
        "_last_lapse_time"
    ],
    inplace=True
)


df["log_days_since_qd"] = np.log1p(
    df["days_since_qd"].clip(lower=0)
)


# ── Lagged features ────────────────────────────────────
grp = df.groupby("account_id", sort=False)

for lag in [1, 2, 3]:
    df[f"sev_lag{lag}"] = (
        grp["craving_severity"]
        .shift(lag)
        .fillna(0)
    )

for lag in [2, 3]:
    df[f"lapse_lag{lag}"] = (
        grp["label"]
        .shift(lag)
        .fillna(0)
    )


EXTRA_FEATS = [
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "severity_change",
    "rolling3_sev_mean",
    "rolling3_sev_std",
    "days_since_last_lapse",
    "log_days_since_qd",
    "sev_lag1",
    "sev_lag2",
    "sev_lag3",
    "lapse_lag2",
    "lapse_lag3"
]

EXP_FEATS = list(
    dict.fromkeys(
        ORIG_FEATS
        + [
            column
            for column in EXTRA_FEATS
            if column in df.columns
        ]
    )
)


# ── Multi-task auxiliary labels ────────────────────────
def phase_fn(days):
    if days < 3:
        return 0
    if days < 14:
        return 1
    return 2


df["phase_label"] = (
    df["days_since_qd"]
    .clip(lower=0)
    .apply(phase_fn)
    .astype(int)
)

grp = df.groupby("account_id", sort=False)

df["next_craving_sev"] = (
    grp["craving_severity"]
    .shift(-1)
)

df["next_craving_change"] = (
    df["next_craving_sev"]
    - df["craving_severity"]
)

# The final entry of each patient has no next craving
df["next_craving_change"] = (
    df["next_craving_change"]
    .fillna(0)
)

df["next_craving_sev"] = (
    df["next_craving_sev"]
    .fillna(0)
)


# ── Eventual lapse within the next 19 days ─────────────
df["eventual_lapse_19d"] = 0.0

print(
    "Computing eventual_lapse_19d..."
)

start_time = time.time()

window_seconds = (
    pd.Timedelta(days=19)
    .total_seconds()
)

for patient_id, patient_group in df.groupby(
    "account_id",
    sort=False
):
    patient_group = patient_group.sort_values(
        "adjusted_craving_record_created"
    )

    indices = patient_group.index.to_numpy()

    times = (
        patient_group["adjusted_craving_record_created"] - EPOCH
    ).dt.total_seconds().to_numpy()

    labels = (
        patient_group["label"]
        .to_numpy()
    )

    lapse_positions = np.flatnonzero(
        labels == 1
    )

    if len(lapse_positions) == 0:
        continue

    for position, row_index in enumerate(indices):
        # Find the first lapse strictly after this entry
        future_lapses = lapse_positions[
            lapse_positions > position
        ]

        if len(future_lapses) == 0:
            continue

        first_future_lapse = future_lapses[0]

        if (
            times[first_future_lapse] - times[position]
            <= window_seconds
        ):
            df.at[
                row_index,
                "eventual_lapse_19d"
            ] = 1.0


elapsed_minutes = (
    time.time() - start_time
) / 60

print(
    f"Eventual-lapse labels completed in "
    f"{elapsed_minutes:.2f} minutes."
)


# ── Fill missing values ────────────────────────────────
ALL_COLS = list(
    dict.fromkeys(
        EXP_FEATS
        + [
            "phase_label",
            "next_craving_change",
            "eventual_lapse_19d"
        ]
    )
)

df[ALL_COLS] = (
    df[ALL_COLS]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .astype(float)
)


# ── Final summary ──────────────────────────────────────
n_orig = len(ORIG_FEATS)
n_exp = len(EXP_FEATS)

print()
print(
    f"Patients              : "
    f"{df['account_id'].nunique():,}"
)
print(
    f"Entries               : "
    f"{len(df):,}"
)
print(
    f"Lapse rate            : "
    f"{df['label'].mean() * 100:.1f}%"
)
print(
    f"Eventual-lapse rate   : "
    f"{df['eventual_lapse_19d'].mean() * 100:.1f}%"
)
print(
    f"Original features     : "
    f"{n_orig}"
)
print(
    f"Expanded features     : "
    f"{n_exp}"
)
print("Cell 2 completed successfully.")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2b — TABLE 1 STATISTICS, computed from the analysed data
# ════════════════════════════════════════════════════════
_n = df.groupby('account_id').size()
_lapses = df.groupby('account_id')['label'].sum()
_t = pd.to_datetime(df['adjusted_craving_record_created'])
_span_days = df.assign(_t=_t).groupby('account_id')['_t'].agg(
    lambda s: max((s.max() - s.min()).total_seconds() / 86400, 1.0))
_active_days = df.assign(_d=_t.dt.date).groupby('account_id')['_d'].nunique()
table1 = pd.DataFrame({'Characteristic': [
    'Total participants', 'Total EMA entries', 'Mean entries per participant (SD)',
    'Median entries per participant (IQR)', 'Range of entries per participant',
    'Overall lapse rate', 'Participants with >=1 lapse', 'Participants with no lapses',
    'Entries per day, mean over first-to-last entry span (median)',
    'Entries per active day, mean (median)'],
 'Value': [
    f"{len(_n)}", f"{len(df):,}", f"{_n.mean():.1f} ({_n.std():.1f})",
    f"{_n.median():.0f} ({_n.quantile(.25):.0f}-{_n.quantile(.75):.0f})",
    f"{_n.min()}-{_n.max()}", f"{100*df['label'].mean():.1f}%",
    f"{(_lapses>0).sum()} ({100*(_lapses>0).mean():.1f}%)",
    f"{(_lapses==0).sum()} ({100*(_lapses==0).mean():.1f}%)",
    f"{(_n/_span_days).mean():.2f} ({(_n/_span_days).median():.2f})",
    f"{(_n/_active_days).mean():.2f} ({(_n/_active_days).median():.2f})"]})
print(table1.to_string(index=False))


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — ALL MODEL CLASSES
# ════════════════════════════════════════════════════════

# ── Dataset utils ─────────────────────────────────────
class PatDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): return self.s[i]

class PatDSExt(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): return self.s[i]

def collate_v(batch):
    xs,ys=zip(*batch)
    lengths=torch.tensor([len(x) for x in xs])
    ml=max(lengths); nc=xs[0].shape[1]
    Xp=torch.zeros(len(xs),ml,nc)
    yp=torch.full((len(xs),ml),float('nan'))
    for i,(x,y) in enumerate(zip(xs,ys)):
        Xp[i,:len(x)]=torch.from_numpy(x).float()
        yp[i,:len(x)]=torch.from_numpy(y).float()
    return Xp,yp,lengths

def collate_ext(batch):
    xs,yl,yph,yn,ye,_=(list(z) for z in zip(*[
        (s[0],s[1],s[2],s[3],s[4],s[5]) for s in batch]))
    lengths=torch.tensor([len(x) for x in xs])
    ml=max(lengths); nc=xs[0].shape[1]
    Xp=torch.zeros(len(xs),ml,nc)
    Yl=torch.full((len(xs),ml),float('nan'))
    Yp=torch.full((len(xs),ml),-1,dtype=torch.long)
    Yn=torch.zeros(len(xs),ml)
    Ye=torch.full((len(xs),ml),float('nan'))
    for i,(x,l,p,n,e) in enumerate(zip(xs,yl,yph,yn,ye)):
        ln=len(x)
        Xp[i,:ln]=torch.from_numpy(x).float()
        Yl[i,:ln]=torch.from_numpy(l).float()
        Yp[i,:ln]=torch.from_numpy(p).long()
        Yn[i,:ln]=torch.from_numpy(n).float()
        Ye[i,:ln]=torch.from_numpy(e).float()
    return Xp,Yl,Yp,Yn,Ye,lengths

# ── Prediction head (shared) ──────────────────────────
class PredHead(nn.Module):
    def __init__(self,hid=64,drop=0.3):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(hid,hid//2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hid//2,1))
    def forward(self,h): return self.net(h).squeeze(-1)

# ── GRU ───────────────────────────────────────────────
class GRUEncoder(nn.Module):
    def __init__(self,nf,hid=64,drop=0.3,n_layers=2):
        super().__init__()
        self.gru=nn.GRU(nf,hid,n_layers,
                         batch_first=True,dropout=drop)
    def forward(self,x,lengths=None):
        self.gru.flatten_parameters()
        if lengths is not None:
            pk=pack_padded_sequence(x,lengths.cpu(),
                batch_first=True,enforce_sorted=False)
            out,_=self.gru(pk)
            out,_=pad_packed_sequence(out,batch_first=True)
        else: out,_=self.gru(x)
        return out

# ── LSTM ──────────────────────────────────────────────
class LSTMEncoder(nn.Module):
    def __init__(self,nf,hid=64,drop=0.3,n_layers=2):
        super().__init__()
        self.lstm=nn.LSTM(nf,hid,n_layers,
                           batch_first=True,dropout=drop)
    def forward(self,x,lengths=None):
        self.lstm.flatten_parameters()
        if lengths is not None:
            pk=pack_padded_sequence(x,lengths.cpu(),
                batch_first=True,enforce_sorted=False)
            out,_=self.lstm(pk)
            out,_=pad_packed_sequence(out,batch_first=True)
        else: out,_=self.lstm(x)
        return out

# ── GRU-D ─────────────────────────────────────────────
class GRUDCell(nn.Module):
    def __init__(self,inp,hid):
        super().__init__()
        gi=inp*2+hid
        self.Wr=nn.Linear(gi,hid)
        self.Wz=nn.Linear(gi,hid)
        self.Wh=nn.Linear(gi,hid)
        self.Wgh=nn.Linear(1,hid)
        self.Wgx=nn.Linear(inp,inp)
        self.h0=nn.Parameter(torch.zeros(hid))
        self.register_buffer('xm',torch.zeros(inp))
    def forward(self,x,m,df_,dg,h):
        gh=torch.exp(-torch.clamp(self.Wgh(dg),min=0))
        ht=gh*h+(1-gh)*self.h0.unsqueeze(0).expand_as(h)
        gx=torch.exp(-torch.clamp(self.Wgx(df_),min=0))
        xi=gx*x+(1-gx)*self.xm.unsqueeze(0).expand_as(x)
        xt=m*x+(1-m)*xi
        inp=torch.cat([xt,m,ht],1)
        r=torch.sigmoid(self.Wr(inp))
        z=torch.sigmoid(self.Wz(inp))
        return (1-z)*ht+z*torch.tanh(
            self.Wh(torch.cat([xt,m,r*ht],1)))

class GRUDEncoder(nn.Module):
    """
    Vectorised GRU-D — no Python timestep loop.
    Uses a standard GRU with time-decay applied
    to the input before each step via batched ops.
    ~20x faster than the cell-loop version.
    """
    def __init__(self,nf,hid=64,drop=0.3,n_layers=2):
        super().__init__()
        self.nf   = nf
        self.hid  = hid
        # learnable decay rates per feature
        self.W_decay = nn.Linear(1, nf, bias=True)
        # input projection: applies decay to features
        self.gru  = nn.GRU(nf*2, hid, n_layers,
                            batch_first=True,
                            dropout=drop)
        self.drop = nn.Dropout(drop)

    def forward(self, x, lengths=None):
        """
        x shape: (B, T, nf*3)
          x[:,:,:nf]      = features
          x[:,:,nf:2*nf]  = missingness mask
          x[:,:,2*nf:3*nf]= time deltas (normalised)
        """
        B, T, _ = x.shape
        nf = self.nf

        feats  = x[:, :, :nf]          # (B,T,nf)
        mask   = x[:, :, nf:2*nf]      # (B,T,nf)
        delta  = x[:, :, 2*nf:3*nf]    # (B,T,nf)

        # decay factor per feature per timestep
        # gamma = exp(-max(0, w*delta + b))
        # shape: (B,T,nf)
        delta_mean = delta.mean(-1, keepdim=True)  # (B,T,1)
        gamma = torch.exp(
            -torch.clamp(
                self.W_decay(delta_mean).expand_as(feats),
                min=0))

        # impute missing values with decayed mean
        # x_imp = mask*x + (1-mask)*gamma*global_mean
        # we use 0 as global mean (features are standardised)
        x_imp = mask * feats + (1 - mask) * gamma * 0.0

        # concatenate imputed features + mask as input
        inp = torch.cat([x_imp, mask], dim=-1)  # (B,T,nf*2)

        self.gru.flatten_parameters()
        if lengths is not None:
            pk  = pack_padded_sequence(
                inp, lengths.cpu(),
                batch_first=True,
                enforce_sorted=False)
            out, _ = self.gru(pk)
            out, _ = pad_packed_sequence(
                out, batch_first=True)
        else:
            out, _ = self.gru(inp)

        out = self.drop(out)
        if lengths is not None:
            for i,l in enumerate(lengths):
                if l < T: out[i,l:] = 0.
        return out
GRUDEncoderLegacy = GRUDEncoder   # the previous implementation
# ── Transformer ───────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self,hid,max_len=700,drop=0.1):
        super().__init__()
        self.drop=nn.Dropout(drop)
        pe=torch.zeros(max_len,hid)
        pos=torch.arange(max_len).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,hid,2).float()*
                       (-np.log(10000)/hid))
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x):
        return self.drop(x+self.pe[:,:x.size(1)])

class TransformerEncoder(nn.Module):
    def __init__(self,nf,hid=64,nhead=4,
                  nlayers=2,drop=0.3):
        super().__init__()
        self.proj=nn.Linear(nf,hid)
        self.pe  =PositionalEncoding(hid,drop=drop)
        enc=nn.TransformerEncoderLayer(
            d_model=hid,nhead=nhead,
            dim_feedforward=hid*2,dropout=drop,
            batch_first=True,norm_first=True)
        self.tfm=nn.TransformerEncoder(enc,nlayers)
    def forward(self,x,lengths=None):
        B,T,_=x.shape
        x=self.pe(self.proj(x))
        mask=torch.triu(torch.ones(T,T,
            device=x.device),diagonal=1).bool()
        pm=None
        if lengths is not None:
            pm=torch.ones(B,T,dtype=torch.bool,
                           device=x.device)
            for i,l in enumerate(lengths): pm[i,:l]=False
        return self.tfm(x,mask=mask,
                         src_key_padding_mask=pm)

# ── SSL model wrapper ─────────────────────────────────
class SSLWrapper(nn.Module):
    def __init__(self,enc,nf,hid=64):
        super().__init__()
        self.enc=enc; h=hid
        self.nh=nn.Sequential(
            nn.Linear(h,h),nn.GELU(),nn.Linear(h,nf))
        self.mh=nn.Sequential(
            nn.Linear(h,h),nn.GELU(),nn.Linear(h,nf))
        self.oh=nn.Sequential(
            nn.Linear(h*2,h),nn.GELU(),nn.Linear(h,1))
    def forward(self,x,lengths=None):
        return self.enc(x,lengths)

# ── Multi-task model wrapper ──────────────────────────
class MTWrapper(nn.Module):
    def __init__(self,enc,nf,hid=64):
        super().__init__()
        self.enc=enc; h=hid
        self.eh=nn.Sequential(
            nn.Linear(h,h//2),nn.GELU(),nn.Linear(h//2,1))
        self.ph=nn.Sequential(
            nn.Linear(h,h//2),nn.GELU(),nn.Linear(h//2,3))
        self.ch=nn.Sequential(
            nn.Linear(h,h//2),nn.GELU(),nn.Linear(h//2,1))
    def forward(self,x,lengths=None):
        return self.enc(x,lengths)

print("✅ Cell 3 done — all model classes defined")


# ════════════════════════════════════════════════════════
# TCN + GRU-TCN IMPLEMENTATION
# Add these classes to Cell 3 after TransformerEncoder
# ════════════════════════════════════════════════════════

# ── Causal Conv Block ─────────────────────────────────
class CausalConv1d(nn.Module):
    """
    Causal convolution: output at t depends only on
    inputs at t, t-1, ..., t-(kernel_size-1)*dilation.
    Achieved by left-padding the input by
    (kernel_size-1)*dilation zeros before convolution
    and removing the rightmost outputs.
    """
    def __init__(self, in_ch, out_ch,
                  kernel_size=3, dilation=1):
        super().__init__()
        self.padding = (kernel_size-1) * dilation
        self.conv    = nn.Conv1d(
            in_ch, out_ch,
            kernel_size = kernel_size,
            dilation    = dilation,
            padding     = 0)

    def forward(self, x):
        # x: (B, C, T)
        x = nn.functional.pad(x, (self.padding, 0))
        return self.conv(x)


class TCNBlock(nn.Module):
    """
    One dilated causal convolutional block.
    Two causal convolutions with:
      - layer normalisation
      - GELU activation
      - dropout
      - residual connection (with 1x1 projection
        if channel dimensions differ)
    """
    def __init__(self, in_ch, out_ch,
                  kernel_size=3, dilation=1,
                  drop=0.3):
        super().__init__()
        self.conv1 = CausalConv1d(
            in_ch, out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(
            out_ch, out_ch, kernel_size, dilation)
        self.norm1 = nn.LayerNorm(out_ch)
        self.norm2 = nn.LayerNorm(out_ch)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(drop)
        # residual projection if channels differ
        self.proj  = (nn.Conv1d(in_ch, out_ch, 1)
                       if in_ch != out_ch else None)

    def forward(self, x):
        # x: (B, T, C) → transpose for conv
        xt = x.transpose(1, 2)          # (B, C, T)
        # first conv
        h  = self.conv1(xt)             # (B, out_ch, T)
        h  = h.transpose(1, 2)          # (B, T, out_ch)
        h  = self.drop(self.act(self.norm1(h)))
        # second conv
        h  = h.transpose(1, 2)          # (B, out_ch, T)
        h  = self.conv2(h)              # (B, out_ch, T)
        h  = h.transpose(1, 2)          # (B, T, out_ch)
        h  = self.drop(self.act(self.norm2(h)))
        # residual
        if self.proj is not None:
            res = self.proj(
                x.transpose(1,2)).transpose(1,2)
        else:
            res = x
        return h + res


class TCNEncoder(nn.Module):
    """
    Temporal Convolutional Network encoder.

    Architecture (Section 4.5):
      4 dilated causal convolutional blocks
      kernel size 3
      dilations: 1, 2, 4, 8
      receptive field: 1+(3-1)*(1+2+4+8)*2 = 61 entries
        covers the median patient sequence length (31)
        and approaches the 75th percentile
      hidden dimension 64 throughout
      dropout 0.3

    Each block: CausalConv → LayerNorm → GELU → Dropout
                CausalConv → LayerNorm → GELU → Dropout
                + residual connection

    Input  : (B, T, nf)
    Output : (B, T, 64)
    """
    def __init__(self, nf, hid=64,
                  kernel_size=3, drop=0.3,
                  dilations=(1,2,4,8)):
        super().__init__()
        self.input_proj = nn.Linear(nf, hid)
        blocks = []
        for d in dilations:
            blocks.append(TCNBlock(
                hid, hid,
                kernel_size = kernel_size,
                dilation    = d,
                drop        = drop))
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x, lengths=None):
        # project input features to hidden dim
        h = self.input_proj(x)      # (B, T, hid)
        # pass through TCN blocks
        h = self.blocks(h)          # (B, T, hid)
        # apply padding mask if lengths provided
        if lengths is not None:
            B, T, _ = h.shape
            for i, l in enumerate(lengths):
                if l < T:
                    h[i, l:] = 0.
        return h


class GRUTCNEncoder(nn.Module):
    """
    Hybrid TCN + GRU encoder.

    Architecture (Section 4.5):
      Front-end: 2 dilated causal TCN blocks
        (dilations 1, 2) for local feature extraction
        receptive field: 1+(3-1)*(1+2)*2 = 13 entries
      Back-end: 1-layer GRU (hidden dim 64)
        for sequential integration across full history
      Hidden dimension 64 throughout
      Dropout 0.3

    Motivation: TCN captures local craving dynamics
    efficiently via parallelisable convolutions;
    GRU integrates these local features into a
    global patient-level trajectory representation
    that respects long-range temporal dependencies.

    Input  : (B, T, nf)
    Output : (B, T, 64)
    """
    def __init__(self, nf, hid=64,
                  kernel_size=3, drop=0.3):
        super().__init__()
        self.input_proj = nn.Linear(nf, hid)
        # 2-block TCN front-end
        self.tcn = nn.Sequential(
            TCNBlock(hid, hid, kernel_size,
                      dilation=1, drop=drop),
            TCNBlock(hid, hid, kernel_size,
                      dilation=2, drop=drop))
        # 1-layer GRU back-end
        self.gru = nn.GRU(hid, hid, num_layers=1,
                           batch_first=True,
                           dropout=0.0)
        self.drop = nn.Dropout(drop)

    def forward(self, x, lengths=None):
        # project input
        h = self.input_proj(x)      # (B, T, hid)
        # local feature extraction via TCN
        h = self.tcn(h)             # (B, T, hid)
        # sequential integration via GRU
        self.gru.flatten_parameters()
        if lengths is not None:
            pk  = pack_padded_sequence(
                h, lengths.cpu().clamp(min=1),
                batch_first=True,
                enforce_sorted=False)
            h, _ = self.gru(pk)
            h, _ = pad_packed_sequence(
                h, batch_first=True)
        else:
            h, _ = self.gru(h)
        h = self.drop(h)
        if lengths is not None:
            B, T, _ = h.shape
            for i, l in enumerate(lengths):
                if l < T:
                    h[i, l:] = 0.
        return h

print("✅ TCNEncoder and GRUTCNEncoder defined")
print(f"  TCN receptive field    : "
      f"{1 + (3-1)*(1+2+4+8)*2} entries")
print(f"  GRU-TCN receptive field: "
      f"{1 + (3-1)*(1+2)*2} entries (TCN front-end)")


# ════════════════════════════════════════════════════════
# GRU-D (Che et al., 2018) — time decay that is actually used
# Layer 1: GRUDCell (defined above, previously unused):
#   hidden-state decay  gamma_h = exp(-max(0, W_h * gap + b))
#   input decay toward the training mean (0 after standardisation)
# Layer 2: standard GRU, matching the 2-layer convention of the other encoders.
# Input layout unchanged: [features | mask | per-feature delta]
# ════════════════════════════════════════════════════════
class GRUDEncoderChe(nn.Module):
    def __init__(self, nf, hid=64, drop=0.3, n_layers=2):
        super().__init__()
        self.nf = nf
        self.cell = GRUDCell(nf, hid)
        self.gru2 = (nn.GRU(hid, hid, n_layers - 1, batch_first=True)
                     if n_layers > 1 else None)
        self.drop = nn.Dropout(drop)

    def forward(self, x, lengths=None):
        B, T, _ = x.shape
        nf = self.nf
        feats = x[:, :, :nf]
        mask  = x[:, :, nf:2 * nf]
        delta = x[:, :, 2 * nf:3 * nf]
        h = self.cell.h0.unsqueeze(0).expand(B, -1).contiguous()
        outs = []
        for t in range(T):
            d_t = delta[:, t, :]
            h = self.cell(feats[:, t, :], mask[:, t, :], d_t,
                          d_t.mean(-1, keepdim=True), h)   # gap is identical across features
            outs.append(h)
        out = torch.stack(outs, dim=1)
        if self.gru2 is not None:
            self.gru2.flatten_parameters()
            out, _ = self.gru2(self.drop(out))
        out = self.drop(out)
        if lengths is not None:
            for i, l in enumerate(lengths):
                if l < T: out[i, l:] = 0.
        return out


GRUDEncoder = GRUDEncoderChe if GRUD_IMPL == 'che2018' else GRUDEncoderLegacy
print(f"✅ GRU-D implementation in use: {GRUD_IMPL}")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — SEQUENCE BUILDERS + STANDARDISATION
# ════════════════════════════════════════════════════════

def build_sequences(df, feat_cols, pid_col='account_id',
                     time_col='adjusted_craving_record_created'):
    seqs=[]
    for pid,g in df.groupby(pid_col):
        g=g.sort_values(time_col).reset_index(drop=True)
        X=g[feat_cols].values.astype(np.float32)
        y=g['label'].values.astype(np.float32)
        seqs.append((X,y,pid))
    return seqs

def build_sequences_grud(df, feat_cols):
    """
    GRU-D sequences: [features | mask | delta_norm]
    Delta = time since last observation of each feature.
    Vectorised: computed with pandas shift, no Python loops.
    """
    seqs = []
    n    = len(feat_cols)

    for pid, g in df.groupby('account_id'):
        g = g.sort_values(
            'adjusted_craving_record_created'
        ).reset_index(drop=True)
        y = g['label'].values.astype(np.float32)

        # features and mask
        feats = g[feat_cols].values.astype(np.float32)
        M     = (~np.isnan(feats)).astype(np.float32)
        feats = np.nan_to_num(feats, nan=0.)

        # time in minutes from first entry
        times = (pd.to_datetime(
            g['adjusted_craving_record_created'])
            - pd.to_datetime(
            g['adjusted_craving_record_created']
            ).iloc[0]
        ).dt.total_seconds().div(60).values.astype(
            np.float32)

        T = len(g)

        # delta: time since previous entry
        # same for all features (entry-level gaps)
        # shape (T,) → broadcast to (T, n)
        gaps  = np.diff(times, prepend=0.).astype(
            np.float32)  # (T,)
        delta = np.broadcast_to(
            gaps[:,None], (T,n)).copy()  # (T,n)

        if GRUD_IMPL == 'legacy':
            # old scaling: divide by this participant's largest gap
            dm = delta.max()
            if dm > 0: delta /= dm
        else:
            # scale without using future entries: log(1+min) / log(1+1 week)
            delta = (np.log1p(delta) / np.log1p(60 * 24 * 7)).astype(np.float32)

        X = np.concatenate([feats, M, delta], axis=1)
        seqs.append((X, y, pid))
    return seqs
    
def build_sequences_ext(df, feat_cols):
    """Extended sequences for multi-task pre-training."""
    seqs=[]
    for pid,g in df.groupby('account_id'):
        g=g.sort_values(
            'adjusted_craving_record_created'
        ).reset_index(drop=True)
        X =g[feat_cols].values.astype(np.float32)
        y =g['label'].values.astype(np.float32)
        yp=g['phase_label'].values.astype(np.int64)
        yn=g['next_craving_change'].values.astype(np.float32)
        ye=g['eventual_lapse_19d'].values.astype(np.float32)
        seqs.append((X,y,yp,yn,ye,pid))
    return seqs

def standardize_fold(train_seqs, test_seqs, nf):
    """Fit on train fold, apply to both. No test leakage."""
    aX=np.vstack([s[0][:,:nf] for s in train_seqs])
    m=aX.mean(0); s=aX.std(0)+1e-6
    def norm(sq):
        X=sq[0].copy(); X[:,:nf]=(sq[0][:,:nf]-m)/s
        return (X,)+sq[1:]
    return ([norm(sq) for sq in train_seqs],
            [norm(sq) for sq in test_seqs], m, s)

def ci95(aucs):
    a=np.array([x for x in aucs if not np.isnan(x)])
    if not len(a): return None,None,None
    se=a.std(ddof=1)/np.sqrt(len(a))
    t=stats.t.ppf(0.975,df=len(a)-1)
    return float(a.mean()),float(a.mean()-t*se),float(a.mean()+t*se)

def youden_threshold(y_true,y_prob):
    fpr,tpr,thresh=roc_curve(y_true,y_prob)
    return thresh[np.argmax(tpr-fpr)]

print("✅ Cell 4 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — SSL + MULTI-TASK PRE-TRAINING FUNCTIONS
# ════════════════════════════════════════════════════════

def ssl_pretrain_fold(train_seqs, enc_class, nf,
                       enc_kwargs=None,
                       epochs=30, bs=8,
                       lr=1e-3, wd=1e-4,
                       mask_rate=0.15):
    """
    SSL pre-training on training fold only (inductive).
    enc_class: GRUEncoder | LSTMEncoder
    """
    if enc_kwargs is None: enc_kwargs={}
    enc=enc_class(nf, **enc_kwargs)
    model=SSLWrapper(enc, nf, 64).to(device)
    opt=torch.optim.Adam(model.parameters(),
                          lr=lr,weight_decay=wd)
    mse=nn.MSELoss(reduction='none')
    bce=nn.BCEWithLogitsLoss()
    dl=DataLoader(PatDS([(s[0],s[1]) for s in train_seqs]),
                  batch_size=bs,shuffle=True,
                  collate_fn=collate_v)
    for ep in range(epochs):
        model.train()
        for Xp,yp,lengths in dl:
            Xp=Xp.to(device); B,T,_=Xp.shape
            Xf=Xp[:,:,:nf]
            opt.zero_grad(); H=model(Xp,lengths)
            loss=torch.tensor(0.,device=device)
            ml=torch.zeros(B,T-1,device=device)
            for i,l in enumerate(lengths):
                ml[i,:min(l-1,T-1)]=1.
            xp=model.nh(H[:,:-1,:]); xt=Xf[:,1:,:]
            loss+=(mse(xp,xt).mean(-1)*ml).sum()/(
                ml.sum()+1e-6)
            mf=(torch.rand(B,T,nf,device=device)<mask_rate)
            Xmc=Xp.clone(); Xmc[:,:,:nf][mf]=0.
            Hm=model(Xmc,lengths); xr=model.mh(Hm)
            if mf.sum()>0:
                loss+=mse(xr[:,:,:nf][mf],Xf[mf]).mean()
            ol=[]
            for i in range(B):
                li=lengths[i].item()
                if li<4: continue
                w=max(2,li//3)
                ta=np.random.randint(0,li-w)
                tb=min(np.random.randint(ta+w,li),li-1)
                ha,hb=H[i,ta,:],H[i,tb,:]
                if np.random.random()>0.5:
                    lg=model.oh(
                        torch.cat([ha,hb]).unsqueeze(0))
                    lb=torch.ones(1,device=device)
                else:
                    lg=model.oh(
                        torch.cat([hb,ha]).unsqueeze(0))
                    lb=torch.zeros(1,device=device)
                ol.append(bce(lg.squeeze(),lb.squeeze()))
            if ol: loss+=0.5*torch.stack(ol).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(
                model.parameters(),1.)
            opt.step()
    return model.enc


def mt_pretrain_fold(train_seqs_ext, enc_class, nf,
                      enc_kwargs=None,
                      l1=0.5, l2=1.0, l3=1.0,
                      epochs=30, bs=8,
                      lr=1e-3, wd=1e-4):
    """
    Multi-task pre-training on training fold only.
    enc_class: GRUEncoder | GRUDEncoder
    """
    if enc_kwargs is None: enc_kwargs={}
    enc=enc_class(nf, **enc_kwargs)
    model=MTWrapper(enc, nf, 64).to(device)
    opt=torch.optim.Adam(model.parameters(),
                          lr=lr,weight_decay=wd)
    bce_=nn.BCEWithLogitsLoss(reduction='none')
    ce_ =nn.CrossEntropyLoss(reduction='none')
    mse_=nn.MSELoss(reduction='none')
    dl=DataLoader(PatDSExt(train_seqs_ext),
                  batch_size=bs,shuffle=True,
                  collate_fn=collate_ext)
    for ep in range(epochs):
        model.train()
        for Xp,Yl,Yp,Yn,Ye,lengths in dl:
            Xp=Xp.to(device); B,T,_=Xp.shape
            Yp=Yp.to(device); Yn=Yn.to(device)
            Ye=Ye.to(device)
            opt.zero_grad(); H=model(Xp,lengths)
            loss=torch.tensor(0.,device=device)
            ml=torch.zeros(B,T,device=device)
            for i,l in enumerate(lengths): ml[i,:l]=1.
            if l1>0:
                ev=model.eh(H).squeeze(-1)
                msk=(~torch.isnan(Ye))&(ml.bool())
                if msk.sum()>0:
                    loss+=l1*bce_(ev[msk],Ye[msk]).mean()
            if l2>0:
                ph=model.ph(H.view(B*T,-1))
                pt=Yp.view(-1)
                mf=ml.view(-1).bool()&(pt>=0)
                if mf.sum()>0:
                    loss+=l2*ce_(ph[mf],pt[mf]).mean()
            if l3>0 and T>1:
                cr=model.ch(H[:,:-1,:]).squeeze(-1)
                ct=Yn[:,1:]
                mc=ml[:,:-1].bool()&(~torch.isnan(ct))
                if mc.sum()>0:
                    loss+=l3*mse_(cr[mc],ct[mc]).mean()
            if loss.item()>0:
                loss.backward()
                nn.utils.clip_grad_norm_(
                    model.parameters(),1.)
                opt.step()
    return model.enc

print("✅ Cell 5 done — pre-training functions defined")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — POOLED CV ENGINE with NESTED HYPERPARAMETER TUNING
# Outer loop: participant-level 10-fold CV (unchanged folds, KFold seed 42).
# Inner loop: within each outer training fold, INNER_VAL_FRAC of the training
#   participants are held out as a validation set. Each hyperparameter setting
#   is trained on the remaining training participants and scored by AUC on the
#   validation participants. The best setting is then refitted on the full
#   outer training fold and evaluated once on the outer test fold.
#   Test-fold participants are never used for tuning.
# Deep models: early stopping on the inner validation AUC selects the number
#   of epochs; the refit on the full training fold uses that number.
# SSL models: the pretrained encoder stays frozen; the head is tuned.
# ════════════════════════════════════════════════════════

# ── hyperparameter grids (first entry = previous fixed setting) ──
ML_GRIDS = {
    'rf':  [{'min_samples_leaf': l, 'max_features': f}
            for l in (5, 1, 20) for f in ('sqrt', 0.3)],
    'xgb': [{'max_depth': d, 'learning_rate': r}
            for d in (6, 3) for r in (0.1, 0.05)],
    'lr':  [{'C': c} for c in (1.0, 0.01, 0.1, 10.0)],
    'svm': [{'C': c} for c in (1.0, 0.1, 10.0)],
}
DL_GRID = [{'lr': l, 'drop': d} for l in (1e-3, 3e-4) for d in (0.3, 0.1)]
DL_DEFAULT = DL_GRID[0]


def save_preds(name, task, fold_idx, pids, y_true, y_prob):
    """One CSV per config / task / fold: participant_id, y_true, y_prob."""
    out = pd.DataFrame({'participant_id': np.asarray(pids, dtype=object),
                        'y_true': np.asarray(y_true, dtype=float).astype(int),
                        'y_prob': np.asarray(y_prob, dtype=float)})
    out.to_csv(os.path.join(SAVE_DIR, 'predictions',
               f"{name}__{task}__fold{fold_idx:02d}.csv"), index=False)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)


def safe_auc(y, p):
    try:
        return float(roc_auc_score(np.asarray(y).astype(int), p))
    except Exception:
        return float('nan')


# ── traditional models ──────────────────────────────────
def flat(seqs, task):
    """Entry-level rows. classify: X_t -> y_t ; next: X_t -> y_{t+1}."""
    X, y, p = [], [], []
    for Xs, ys, pid in seqs:
        if task == 'classify':
            X.append(Xs); y.append(ys); p.append(np.repeat(pid, len(ys)))
        elif len(ys) >= 2:
            X.append(Xs[:-1]); y.append(ys[1:]); p.append(np.repeat(pid, len(ys) - 1))
    return np.vstack(X), np.concatenate(y), np.concatenate(p)


def fit_ml(mtype, hp, X, y):
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X); y = y.astype(int)
    pos = max((y == 1).sum(), 1); neg = (y == 0).sum()
    if mtype == 'rf':
        clf = RandomForestClassifier(
            n_estimators=500, max_depth=None, class_weight='balanced',
            min_samples_leaf=hp.get('min_samples_leaf', 5),
            max_features=hp.get('max_features', 'sqrt'),
            n_jobs=-1, random_state=42)
    elif mtype == 'xgb':
        clf = xgb.XGBClassifier(
            n_estimators=500, scale_pos_weight=neg / pos,
            learning_rate=hp.get('learning_rate', 0.1), max_depth=hp.get('max_depth', 6),
            subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
            n_jobs=-1, random_state=42, verbosity=0)
    elif mtype == 'lr':
        clf = LogisticRegression(C=hp.get('C', 1.0), max_iter=1000,
                                 class_weight='balanced', solver='lbfgs', random_state=42)
    elif mtype == 'svm':
        if len(Xs) > 5000:
            idx_ = np.random.default_rng(42).choice(len(Xs), 5000, replace=False)
            Xs, y = Xs[idx_], y[idx_]
        clf = SVC(kernel='rbf', C=hp.get('C', 1.0), gamma='scale',
                  class_weight='balanced', probability=True, random_state=42)
    clf.fit(Xs, y)
    return lambda Xnew: clf.predict_proba(scaler.transform(Xnew))[:, 1]


# ── deep models ─────────────────────────────────────────
def build_encoder(mtype, inp_nf, nf, drop):
    if mtype == 'gru':         return GRUEncoder(inp_nf, 64, drop=drop)
    if mtype == 'lstm':        return LSTMEncoder(inp_nf, 64, drop=drop)
    if mtype == 'grud':        return GRUDEncoder(nf, 64, drop=drop)
    if mtype == 'transformer': return TransformerEncoder(inp_nf, 64, drop=drop)
    if mtype == 'tcn':         return TCNEncoder(inp_nf, 64, drop=drop)
    if mtype == 'gru_tcn':     return GRUTCNEncoder(inp_nf, 64, drop=drop)
    raise ValueError(mtype)


def predict_dl(enc, head, seqs, task):
    enc.eval(); head.eval()
    probs, labels, pids = [], [], []
    with torch.no_grad():
        for X, y, pid in seqs:
            if task == 'next' and len(y) < 2:
                continue
            H = enc(torch.from_numpy(X).float().unsqueeze(0).to(device))
            if task == 'next':
                pr = torch.sigmoid(head(H[:, :-1, :]).squeeze(0)).cpu().numpy(); gt = y[1:]
            else:
                pr = torch.sigmoid(head(H).squeeze(0)).cpu().numpy(); gt = y
            probs.extend(np.atleast_1d(pr).tolist()); labels.extend(gt.tolist())
            pids.extend([pid] * len(gt))
    return np.array(probs), np.array(labels), pids


def train_dl(enc, head, frozen, train_s, task, epochs, lr, wd, bs,
             val_s=None, patience=None):
    """Trains in place. With val_s: early stopping on validation AUC;
    returns (best_auc, best_epoch). Without: returns (nan, epochs)."""
    y_all = np.concatenate([s[1] for s in train_s])
    pw = torch.tensor([(y_all == 0).sum() / max((y_all == 1).sum(), 1)]
                      ).clamp(max=20).to(device)
    lf = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')
    params = list(head.parameters()) + ([] if frozen else list(enc.parameters()))
    opt = torch.optim.Adam(params, lr=lr, weight_decay=wd)
    dl = DataLoader(PatDS([(s[0], s[1]) for s in train_s]),
                    batch_size=bs, shuffle=True, collate_fn=collate_v)
    best_auc, best_ep, since = -1.0, epochs, 0
    for ep in range(epochs):
        enc.train(); head.train()
        if frozen: enc.eval()
        for Xp, yp, lengths in dl:
            Xp, yp = Xp.to(device), yp.to(device)
            if frozen:
                with torch.no_grad(): H = enc(Xp, lengths)
            else:
                H = enc(Xp, lengths)
            pr, tg = ((head(H[:, :-1, :]), yp[:, 1:]) if task == 'next' else (head(H), yp))
            mask = ~torch.isnan(tg)
            if mask.sum() == 0: continue
            loss = lf(pr[mask], tg[mask]).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params, 1.)
            opt.step()
        if val_s is not None:
            p, l, _ = predict_dl(enc, head, val_s, task)
            a = safe_auc(l, p)
            if a > best_auc: best_auc, best_ep, since = a, ep + 1, 0
            else:
                since += 1
                if patience and since >= patience: break
    return (best_auc, best_ep) if val_s is not None else (float('nan'), epochs)


# ── main CV loop ───────────────────────────────────────
def run_cv(df, model_cfg, n_folds=10, ft_epochs=30, bs=32,
           ssl_epochs=30, ssl_bs=8, lr=1e-3, wd=1e-4, random_seed=42):
    name, feat_cols, mtype = model_cfg['name'], model_cfg['feat_cols'], model_cfg['model_type']
    save_path = f"{SAVE_DIR}/{name}.json"
    results = {'classify': [], 'next': []}
    hparams = {'classify': [], 'next': []}
    start = 0
    if os.path.exists(save_path):
        d = json.load(open(save_path))
        saved = d.get('fold_aucs', {})
        start = min(len(saved.get('classify', [])), len(saved.get('next', [])))
        results = {t: saved.get(t, [])[:start] for t in results}
        hparams = {t: d.get('hparams', {}).get(t, [None] * start)[:start] for t in hparams}
        if start == n_folds:
            print(f"  ✅ Complete: {name}"); return results
        if start > 0: print(f"  Resuming {name} from fold {start+1}")

    nf = len(feat_cols)
    is_grud = mtype in ('grud', 'ssl_grud')
    is_ml = mtype in ('rf', 'xgb', 'lr', 'svm')
    is_ssl = mtype.startswith('ssl_')
    all_seqs = build_sequences_grud(df, feat_cols) if is_grud else build_sequences(df, feat_cols)
    inp_nf = nf * 3 if is_grud else nf

    pid_arr = np.array([s[2] for s in all_seqs])
    splits = list(KFold(n_splits=n_folds, shuffle=True, random_state=random_seed).split(pid_arr))
    fa_path = os.path.join(SAVE_DIR, 'fold_assignment.json')
    if not os.path.exists(fa_path):
        json.dump({str(k): sorted(map(str, pid_arr[te])) for k, (_, te) in enumerate(splits)},
                  open(fa_path, 'w'), indent=1)

    tune = (TUNE == 'all') or (TUNE == 'traditional' and is_ml)
    t_cv = time.time()
    print(f"\n  {name} | {mtype} | {nf} feats | tuning={'on' if tune else 'off'}")

    for fold_idx, (tr_idx, te_idx) in enumerate(splits):
        if fold_idx < start: continue
        set_seed(SEED + fold_idx)
        train_raw = [all_seqs[i] for i in tr_idx]
        test_raw = [all_seqs[i] for i in te_idx]
        train_s, test_s, _, _ = standardize_fold(train_raw, test_raw, nf)

        # inner split of the TRAINING participants (never the test fold)
        perm = np.random.default_rng(SEED + fold_idx).permutation(len(train_raw))
        n_val = max(1, int(round(INNER_VAL_FRAC * len(train_raw))))
        itr_s, ival_s, _, _ = standardize_fold([train_raw[i] for i in perm[n_val:]],
                                               [train_raw[i] for i in perm[:n_val]], nf)

        ssl_enc = None
        if is_ssl:
            enc_cls = {'ssl_gru': GRUEncoder, 'ssl_lstm': LSTMEncoder,
                       'ssl_grud': GRUDEncoder}[mtype]
            ssl_enc = ssl_pretrain_fold(train_s, enc_cls, nf,
                                        enc_kwargs={'hid': 64, 'drop': 0.3, 'n_layers': 2},
                                        epochs=ssl_epochs, bs=ssl_bs, lr=lr, wd=wd)

        for task in ['classify', 'next']:
            if is_ml:
                grid = ML_GRIDS[mtype] if tune else [ML_GRIDS[mtype][0]]
                best = grid[0]
                if len(grid) > 1:
                    Xi, yi, _ = flat(itr_s, task); Xv, yv, _ = flat(ival_s, task)
                    scores = [safe_auc(yv, fit_ml(mtype, hp, Xi, yi)(Xv)) for hp in grid]
                    if not np.all(np.isnan(scores)):   # no lapses in validation -> keep default
                        best = grid[int(np.nanargmax(scores))]
                Xtr, ytr, _ = flat(train_s, task); Xte, yte, pte = flat(test_s, task)
                probs = fit_ml(mtype, best, Xtr, ytr)(Xte)
                save_preds(name, task, fold_idx, pte, yte, probs)
                results[task].append(safe_auc(yte, probs)); hparams[task].append(best)
                continue

            def fresh(hp):
                set_seed(SEED + fold_idx)
                enc = (copy.deepcopy(ssl_enc) if is_ssl
                       else build_encoder(mtype, inp_nf, nf, hp['drop'])).to(device)
                if is_ssl:
                    for p_ in enc.parameters(): p_.requires_grad = False
                return enc, PredHead(64, drop=hp['drop']).to(device)

            grid = DL_GRID if tune else [DL_DEFAULT]
            if tune:
                trials = []
                for hp in grid:
                    enc, head = fresh(hp)
                    a, ep = train_dl(enc, head, is_ssl, itr_s, task, ft_epochs, hp['lr'], wd, bs,
                                     val_s=ival_s, patience=PATIENCE)
                    trials.append((a, ep, hp))
                if all(np.isnan(t[0]) or t[0] < 0 for t in trials):   # no lapses in validation
                    best = dict(DL_DEFAULT, epochs=ft_epochs)
                else:
                    a_best, ep_best, best = max(trials, key=lambda t: np.nan_to_num(t[0], nan=-1))
                    best = dict(best, epochs=int(ep_best))
            else:
                best = dict(DL_DEFAULT, epochs=ft_epochs)

            enc, head = fresh(best)
            train_dl(enc, head, is_ssl, train_s, task, best['epochs'], best['lr'], wd, bs)
            probs, labels, pids_ = predict_dl(enc, head, test_s, task)
            save_preds(name, task, fold_idx, pids_, labels, probs)
            results[task].append(safe_auc(labels, probs)); hparams[task].append(best)

        elapsed = (time.time() - t_cv) / 60
        eta = elapsed / (fold_idx - start + 1) * (n_folds - fold_idx - 1)
        print(f"  Fold {fold_idx+1:2d}/{n_folds}  cls={results['classify'][-1]:.4f}  "
              f"next={results['next'][-1]:.4f}  ETA:{eta:.0f}min")
        json.dump({'fold_aucs': results, 'hparams': hparams, 'model': name,
                   'inductive_ssl': True, 'grud_impl': GRUD_IMPL, 'seed': SEED,
                   'tuning': TUNE if tune else 'off', 'fold_normalisation': True},
                  open(save_path, 'w'), default=str)

    print(f"\n  {name} RESULTS:")
    for task in ['classify', 'next']:
        m, lo, hi = ci95(results[task])
        if m: print(f"  {task:<12} {m:.4f} [{lo:.4f}–{hi:.4f}]")
    return results

print("✅ Cell 6 done — CV engine with nested tuning ready")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — ALL 24 MODEL–FEATURE CONFIGURATIONS
# 12 architectures x {original, expanded}.
# ════════════════════════════════════════════════════════

MODEL_CONFIGS = [
    {'name': 'SSLGRU_exp',    'feat_cols': EXP_FEATS,  'model_type': 'ssl_gru'},
    {'name': 'SSLGRUD_exp',   'feat_cols': EXP_FEATS,  'model_type': 'ssl_grud'},
    {'name': 'SSLGRU_orig',   'feat_cols': ORIG_FEATS, 'model_type': 'ssl_gru'},
    {'name': 'GRUD_exp',      'feat_cols': EXP_FEATS,  'model_type': 'grud'},
    {'name': 'RF_exp',        'feat_cols': EXP_FEATS,  'model_type': 'rf'},
    {'name': 'SSLGRUD_orig',  'feat_cols': ORIG_FEATS, 'model_type': 'ssl_grud'},
    {'name': 'GRUD_orig',     'feat_cols': ORIG_FEATS, 'model_type': 'grud'},
    {'name': 'GRUTCN_orig',   'feat_cols': ORIG_FEATS, 'model_type': 'gru_tcn'},
    {'name': 'LR_exp',        'feat_cols': EXP_FEATS,  'model_type': 'lr'},
    {'name': 'XGB_exp',       'feat_cols': EXP_FEATS,  'model_type': 'xgb'},
    {'name': 'TCN_orig',      'feat_cols': ORIG_FEATS, 'model_type': 'tcn'},
    {'name': 'GRU_orig',      'feat_cols': ORIG_FEATS, 'model_type': 'gru'},
    {'name': 'GRU_exp',       'feat_cols': EXP_FEATS,  'model_type': 'gru'},
    {'name': 'LSTM_orig',     'feat_cols': ORIG_FEATS, 'model_type': 'lstm'},
    {'name': 'SVM_exp',       'feat_cols': EXP_FEATS,  'model_type': 'svm'},
    {'name': 'RF_orig',       'feat_cols': ORIG_FEATS, 'model_type': 'rf'},
    {'name': 'XGB_orig',      'feat_cols': ORIG_FEATS, 'model_type': 'xgb'},
    {'name': 'LR_orig',       'feat_cols': ORIG_FEATS, 'model_type': 'lr'},
    {'name': 'SVM_orig',      'feat_cols': ORIG_FEATS, 'model_type': 'svm'},
    # added in version 2
    {'name': 'LSTM_exp',      'feat_cols': EXP_FEATS,  'model_type': 'lstm'},
    {'name': 'TCN_exp',       'feat_cols': EXP_FEATS,  'model_type': 'tcn'},
    {'name': 'GRUTCN_exp',    'feat_cols': EXP_FEATS,  'model_type': 'gru_tcn'},
    {'name': 'TFM_orig',      'feat_cols': ORIG_FEATS, 'model_type': 'transformer'},
    {'name': 'TFM_exp',       'feat_cols': EXP_FEATS,  'model_type': 'transformer'},
]

assert RUN_ONLY is not None or len(MODEL_CONFIGS) == 24, f"Expected 24 configs, got {len(MODEL_CONFIGS)}"

if RUN_ONLY is not None:
    MODEL_CONFIGS = [c for c in MODEL_CONFIGS if c['name'] in RUN_ONLY]
    print(f"Running only: {[c['name'] for c in MODEL_CONFIGS]}")

ML_NAMES = [c['name'] for c in MODEL_CONFIGS
            if c['model_type'] in ('rf', 'xgb', 'lr', 'svm')]
DL_NAMES = [c['name'] for c in MODEL_CONFIGS
            if c['model_type'] not in ('rf', 'xgb', 'lr', 'svm')]

print(f"Traditional ML configs ({len(ML_NAMES)}): {ML_NAMES}")
print(f"Deep learning / SSL configs ({len(DL_NAMES)}): {DL_NAMES}")

all_results = {}


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — RUN BATCH 1: TRADITIONAL ML (8 configs)
# Fast — no GPU epochs required.
# ════════════════════════════════════════════════════════

for cfg in MODEL_CONFIGS:
    if cfg['name'] not in ML_NAMES:
        continue
    print(f"\n{'═'*55}")
    print(f"  {cfg['name']}")
    print(f"{'═'*55}")
    r = run_cv(df, cfg, n_folds=N_FOLDS, ft_epochs=FT_EPOCHS, bs=32,
               lr=1e-3, wd=1e-4)
    all_results[cfg['name']] = r

print("\n✅ Batch 1 (ML) complete")
print(f"{'model':<15} {'classify AUC':>14} {'next AUC':>14}")
for name in ML_NAMES:
    r = all_results.get(name)
    if r is None: continue
    cm = float(np.nanmean(r.get('classify', [])))
    nm = float(np.nanmean(r.get('next', [])))
    print(f"{name:<15} {cm:>14.4f} {nm:>14.4f}")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 9 — RUN BATCH 2: DEEP LEARNING / SSL (16 configs)
# Slower — GPU recommended. Includes SSL pre-training,
# supervised sequence models, and TCN/GRU-TCN.
# Run this after reviewing Batch 1 results.
# ════════════════════════════════════════════════════════

for cfg in MODEL_CONFIGS:
    if cfg['name'] not in DL_NAMES:
        continue
    print(f"\n{'═'*55}")
    print(f"  {cfg['name']}")
    print(f"{'═'*55}")
    r = run_cv(df, cfg, n_folds=N_FOLDS, ft_epochs=FT_EPOCHS, bs=32,
               ssl_epochs=SSL_EPOCHS, ssl_bs=8, lr=1e-3, wd=1e-4)
    all_results[cfg['name']] = r

print("\n✅ Batch 2 (DL/SSL) complete")
print(f"{'model':<15} {'classify AUC':>14} {'next AUC':>14}")
for name in DL_NAMES:
    r = all_results.get(name)
    if r is None: continue
    cm = float(np.nanmean(r.get('classify', [])))
    nm = float(np.nanmean(r.get('next', [])))
    print(f"{name:<15} {cm:>14.4f} {nm:>14.4f}")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 10 — COMBINED RESULTS TABLE (all 24 configs)
# Reports mean AUC and 95% CI per fold-set, for both tasks,
# in the same format as the manuscript's Table 2.
# ════════════════════════════════════════════════════════

def ci95(aucs):
    a = np.array([x for x in aucs if not np.isnan(x)])
    if not len(a): return None, None, None
    se = a.std(ddof=1) / np.sqrt(len(a))
    t  = stats.t.ppf(0.975, df=len(a) - 1)
    return float(a.mean()), float(a.mean() - t*se), float(a.mean() + t*se)

rows = []
for cfg in MODEL_CONFIGS:
    name = cfg['name']
    r = all_results.get(name)
    if r is None:
        continue
    cm, clo, chi = ci95(r.get('classify', []))
    nm, nlo, nhi = ci95(r.get('next', []))
    rows.append({
        'model': name,
        'model_type': cfg['model_type'],
        'features': 'expanded' if cfg['feat_cols'] is EXP_FEATS else 'original',
        'classify_AUC': round(cm, 4) if cm else None,
        'classify_CI': f"[{clo:.3f}–{chi:.3f}]" if cm else '—',
        'next_AUC': round(nm, 4) if nm else None,
        'next_CI': f"[{nlo:.3f}–{nhi:.3f}]" if nm else '—',
    })

results_table = pd.DataFrame(rows).sort_values('classify_AUC', ascending=False)
print(results_table.to_string(index=False))

results_table.to_csv(f"{SAVE_DIR}/objective1_full_table2.csv", index=False)
print(f"\nSaved → {SAVE_DIR}/objective1_full_table2.csv")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 11 — PER-PARTICIPANT AUC (Table 2b) + PAIRED TESTS
# Uses the saved predictions and fold AUCs. No retraining.
# ════════════════════════════════════════════════════════
import glob, itertools

# Pre-specified comparisons (fewer tests -> Holm correction is less severe)
# 12 x original-vs-expanded + 2 time-decay + 3 top-vs-RF = 17 comparisons per task.
# With 10 folds the smallest Holm-adjusted p is 0.002 x 17 = 0.034, so keep this list short.
FEATURE_PAIRS = [(f'{m}_exp', f'{m}_orig') for m in
                 ['RF', 'XGB', 'LR', 'SVM', 'LSTM', 'GRU', 'GRUD', 'TCN', 'GRUTCN', 'TFM',
                  'SSLGRU', 'SSLGRUD']]
PAIRS = FEATURE_PAIRS + [
    # does explicit time decay help?
    ('GRUD_orig', 'GRU_orig'), ('GRUD_exp', 'GRU_exp'),
    # leading SSL configurations vs the best traditional model
    ('SSLGRU_exp', 'RF_exp'), ('SSLGRU_orig', 'RF_exp'), ('SSLGRUD_exp', 'RF_exp'),
]
NB_RATIO = 1 / (N_FOLDS - 1)   # test/train ratio for the Nadeau-Bengio correction


def holm(p):
    p = np.asarray(p, float); adj = np.empty(len(p)); run = 0.0
    for r, i in enumerate(np.argsort(p)):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj


# fold AUCs from the saved JSON files
fold = {}
for cfg in MODEL_CONFIGS:
    pth = f"{SAVE_DIR}/{cfg['name']}.json"
    if os.path.exists(pth):
        fold[cfg['name']] = json.load(open(pth))['fold_aucs']

# 1) per-participant AUC (participants with 0% or 100% lapses in held-out entries excluded)
pp_rows = []
for name in fold:
    for task in ['classify', 'next']:
        files = sorted(glob.glob(f"{SAVE_DIR}/predictions/{name}__{task}__fold*.csv"))
        if not files: continue
        preds = pd.concat(pd.read_csv(f) for f in files)
        aucs = np.array([roc_auc_score(g['y_true'], g['y_prob'])
                         for _, g in preds.groupby('participant_id')
                         if g['y_true'].nunique() == 2])
        pp_rows.append({'model': name, 'task': task,
                        'pooled_AUC_mean_of_folds': round(np.nanmean(fold[name][task]), 3),
                        'per_participant_median': round(float(np.median(aucs)), 3),
                        'IQR_low': round(float(np.percentile(aucs, 25)), 3),
                        'IQR_high': round(float(np.percentile(aucs, 75)), 3),
                        'n_included': len(aucs)})
per_participant = pd.DataFrame(pp_rows)

# 2) paired fold-level tests
test_rows = []
for task in ['classify', 'next']:
    for a, b in PAIRS:
        if a not in fold or b not in fold: continue
        d = np.array(fold[a][task]) - np.array(fold[b][task])
        d = d[~np.isnan(d)]; k = len(d)
        if k < 2: continue
        w_p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        sd = d.std(ddof=1)
        nb_p = (2 * stats.t.sf(abs(d.mean() / np.sqrt((1/k + NB_RATIO) * sd**2)), df=k-1)
                if sd > 0 else np.nan)
        test_rows.append({'task': task, 'A': a, 'B': b, 'k_folds': k,
                          'mean_diff_A_minus_B': round(d.mean(), 4),
                          'wilcoxon_p': w_p, 'nadeau_bengio_p': nb_p})
tests = pd.DataFrame(test_rows)
if len(tests):
    tests['wilcoxon_p_holm'] = tests.groupby('task')['wilcoxon_p'].transform(holm)

per_participant.to_csv(f"{SAVE_DIR}/per_participant_auc.csv", index=False)
tests.to_csv(f"{SAVE_DIR}/paired_tests.csv", index=False)
print("Per-participant AUC (Table 2b)"); print(per_participant.to_string(index=False))
print("\nPaired tests (Holm applied within each task)"); print(tests.round(4).to_string(index=False))
print(f"\nSaved to {SAVE_DIR}/per_participant_auc.csv and {SAVE_DIR}/paired_tests.csv")


# 3) hyperparameters chosen in each fold (report the most frequent choice per model in Methods)
hp_rows = []
for cfg in MODEL_CONFIGS:
    pth = f"{SAVE_DIR}/{cfg['name']}.json"
    if not os.path.exists(pth): continue
    hp = json.load(open(pth)).get('hparams', {})
    for task in ['classify', 'next']:
        chosen = [json.dumps(h, sort_keys=True) if isinstance(h, dict) else str(h)
                  for h in hp.get(task, [])]
        if not chosen: continue
        vc = pd.Series(chosen).value_counts()
        hp_rows.append({'model': cfg['name'], 'task': task, 'most_frequent': vc.index[0],
                        'folds_chosen': f"{vc.iloc[0]}/{len(chosen)}",
                        'n_distinct_choices': len(vc)})
hp_summary = pd.DataFrame(hp_rows)
hp_summary.to_csv(f"{SAVE_DIR}/chosen_hyperparameters.csv", index=False)
print("\nChosen hyperparameters (most frequent per model and task)")
print(hp_summary.to_string(index=False))


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 12 — DO THE EXPANDED FEATURES HELP WITHIN PERSONS?
# Paired per-participant comparison from the saved predictions (no retraining).
# ════════════════════════════════════════════════════════
def participant_aucs(name, task):
    files = sorted(glob.glob(f"{SAVE_DIR}/predictions/{name}__{task}__fold*.csv"))
    preds = pd.concat(pd.read_csv(f) for f in files)
    return pd.Series({pid: roc_auc_score(g['y_true'], g['y_prob'])
                      for pid, g in preds.groupby('participant_id')
                      if g['y_true'].nunique() == 2})

rows = []
for task in ['classify', 'next']:
    for a, b in FEATURE_PAIRS:
        if a not in fold or b not in fold: continue
        pa, pb = participant_aucs(a, task), participant_aucs(b, task)
        common = pa.index.intersection(pb.index)
        d = (pa[common] - pb[common]).values
        rows.append({'task': task, 'A': a, 'B': b, 'n_participants': len(d),
                     'median_A': round(float(np.median(pa[common])), 3),
                     'median_B': round(float(np.median(pb[common])), 3),
                     'median_diff_A_minus_B': round(float(np.median(d)), 3),
                     'share_A_better': round(float((d > 0).mean()), 2),
                     'wilcoxon_p': stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0})
within = pd.DataFrame(rows)
within['wilcoxon_p_holm'] = within.groupby('task')['wilcoxon_p'].transform(holm)
within.to_csv(f"{SAVE_DIR}/within_person_feature_effect.csv", index=False)
print(within.round(4).to_string(index=False))


## Self-supervised heads: early-stopping check (Supplement S3)

In [ ]:
C13 = r'''# ════════════════════════════════════════════════════════
# CELL 13 — SSL HEADS: WHICH EPOCH DID TUNING CHOOSE IN EACH FOLD? (reads the saved v2 results; no training)
# ════════════════════════════════════════════════════════
rows = []
for cfg in MODEL_CONFIGS:
    if cfg['model_type'] in ('rf', 'xgb', 'lr', 'svm'): continue
    path = os.path.join(SAVE_DIR, f"{cfg['name']}.json")
    if not os.path.exists(path): continue
    hp = json.load(open(path)).get('hparams', {})
    for task in ['classify', 'next']:
        eps = []
        for h in hp.get(task, []):
            if isinstance(h, str):
                try: h = json.loads(h.replace("'", '"'))
                except Exception: h = None
            if isinstance(h, dict) and 'epochs' in h: eps.append(int(h['epochs']))
        if eps:
            rows.append(dict(model=cfg['name'], task=task, folds=len(eps), epochs_per_fold=eps,
                             at_cap=f"{sum(e >= FT_EPOCHS for e in eps)}/{len(eps)}", median_epoch=float(np.median(eps))))
EPOCHS_CHOSEN = pd.DataFrame(rows)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 80)
print(f"Epoch chosen by early stopping in each fold (cap = {FT_EPOCHS}; 'at_cap' = best validation AUC at the last allowed epoch)")
print(EPOCHS_CHOSEN.to_string(index=False))
'''

C14 = r'''# ════════════════════════════════════════════════════════
# CELL 14 — SSL HEADS: EARLY-STOPPING CHECK WITH A HIGHER EPOCH LIMIT
# Same outer folds (KFold seed 42) and the same inner validation participants as v2. Per fold, the SSL encoder is
# pretrained exactly as in v2 (seed SEED + fold), then frozen. Because the encoder is frozen and in eval mode, its
# outputs are computed once and cached; training the head on the cached outputs is identical to v2's training.
# Three arms on the SAME encoder, so the comparison does not depend on reproducing the original run:
#   A  'v2'          max 30 epochs, patience 5; hyperparameters + epoch count chosen on validation; head refitted on
#                    the full training fold for that many epochs (the v2 protocol)
#   B  'long_ckpt'   max ES_MAX_EPOCHS, patience ES_PATIENCE; hyperparameters chosen on validation; the head CHECKPOINT
#                    with the best validation AUC (trained on the inner-training participants) is evaluated  [primary]
#   C  'long_refit'  as B, but refitted on the full training fold for B's best epoch count (isolates the refit step)
# Only validation participants are used to choose the hyperparameters and the stopping point; test folds are touched once.
# Validation history (train loss, validation loss, validation AUC per epoch) is saved for every fold × task × setting.
# ════════════════════════════════════════════════════════
ES_CONFIGS    = ['SSLGRU_exp', 'SSLGRUD_exp', 'SSLGRU_orig', 'SSLGRUD_orig']   # start with ['SSLGRU_exp'] if time is short
ES_MAX_EPOCHS = 200
ES_PATIENCE   = 20
ES_MIN_DELTA  = 0.0          # an improvement must exceed this to reset patience (0 = any improvement, as in v2)
ES_DIR = os.path.join(SAVE_DIR, 'ssl_es_check'); os.makedirs(ES_DIR, exist_ok=True)
if QUICK_TEST: ES_MAX_EPOCHS, ES_PATIENCE = 6, 2


def es_embed(enc, seqs):
    """Frozen encoder outputs, one array (T, 64) per participant (identical to v2's frozen forward pass)."""
    enc.eval(); out = []
    with torch.no_grad():
        for X, y, pid in seqs:
            out.append((enc(torch.from_numpy(X).float().unsqueeze(0).to(device))[0].cpu().numpy(), y, pid))
    return out


def es_collate(batch):
    hs, ys = zip(*batch); T = max(len(h) for h in hs)
    H = torch.zeros(len(hs), T, hs[0].shape[1]); Y = torch.full((len(hs), T), float('nan'))
    for i, (h, y) in enumerate(zip(hs, ys)):
        H[i, :len(h)] = torch.from_numpy(h).float(); Y[i, :len(y)] = torch.from_numpy(y).float()
    return H, Y


def es_predict(head, emb, task):
    head.eval(); P, L, I = [], [], []
    with torch.no_grad():
        for H, y, pid in emb:
            if task == 'next' and len(y) < 2: continue
            h = torch.from_numpy(H).float().unsqueeze(0).to(device)
            pr = torch.sigmoid(head(h[:, :-1, :] if task == 'next' else h)).squeeze(0).cpu().numpy()
            gt = y[1:] if task == 'next' else y
            P += np.atleast_1d(pr).tolist(); L += gt.tolist(); I += [pid] * len(gt)
    return np.array(P), np.array(L), I


def es_train(emb_tr, task, hp, seed, max_epochs, emb_val=None, patience=None, min_delta=0.0, wd=1e-4, bs=32):
    """v2's head training (same loss, class weight, optimiser, clipping, batch size) with a full history and the
    best-validation checkpoint. Without emb_val it trains exactly max_epochs (refit)."""
    set_seed(seed)
    y_all = np.concatenate([s[1] for s in emb_tr])
    pw = torch.tensor([(y_all == 0).sum() / max((y_all == 1).sum(), 1)]).clamp(max=20).to(device)
    lf = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')
    head = PredHead(64, drop=hp['drop']).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=hp['lr'], weight_decay=wd)
    dl = DataLoader(PatDS([(s[0], s[1]) for s in emb_tr]), batch_size=bs, shuffle=True, collate_fn=es_collate)
    hist = dict(train_loss=[], val_loss=[], val_auc=[]); best = (-1.0, 0, copy.deepcopy(head.state_dict())); since = 0
    for ep in range(max_epochs):
        head.train(); tl = []
        for H, Y in dl:
            H, Y = H.to(device), Y.to(device)
            pr, tg = (head(H[:, :-1, :]), Y[:, 1:]) if task == 'next' else (head(H), Y)
            m = ~torch.isnan(tg)
            if m.sum() == 0: continue
            loss = lf(pr[m], tg[m]).mean(); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(list(head.parameters()), 1.); opt.step(); tl.append(loss.item())
        hist['train_loss'].append(float(np.mean(tl)) if tl else float('nan'))
        if emb_val is None: continue
        p, l, _ = es_predict(head, emb_val, task)
        a = safe_auc(l, p); pc = np.clip(p, 1e-7, 1 - 1e-7)
        hist['val_auc'].append(a); hist['val_loss'].append(float(-np.mean(l * np.log(pc) + (1 - l) * np.log(1 - pc))))
        if not np.isnan(a) and a > best[0] + min_delta:
            best = (a, ep + 1, copy.deepcopy(head.state_dict())); since = 0
        else:
            since += 1
            if patience and since >= patience: break
    if emb_val is None: return head, hist
    head.load_state_dict(best[2])
    return head, dict(hist, best_auc=best[0], best_epoch=best[1], stopped_epoch=len(hist['val_auc']))


def es_select(emb_itr, emb_val, task, seed, max_epochs, patience, min_delta):
    """Grid search over v2's DL_GRID on validation; returns (best checkpoint head, chosen hp, its history, all histories)."""
    trials = []
    for hp in DL_GRID:
        head, h = es_train(emb_itr, task, hp, seed, max_epochs, emb_val, patience, min_delta)
        trials.append((h['best_auc'], hp, head, h))
    if all(t[0] < 0 for t in trials):                       # no usable validation AUC: v2's fallback
        hp = DL_DEFAULT; head, h = es_train(emb_itr, task, hp, seed, max_epochs)
        return head, hp, dict(h, best_epoch=max_epochs, best_auc=float('nan')), trials
    a, hp, head, h = max(trials, key=lambda t: t[0])
    return head, hp, h, trials


es_cfgs = [c for c in MODEL_CONFIGS if c['name'] in ES_CONFIGS]
t0 = time.time()
for cfg in es_cfgs:
    name, feat_cols, mtype = cfg['name'], cfg['feat_cols'], cfg['model_type']
    nf = len(feat_cols); is_grud = mtype == 'ssl_grud'
    all_seqs = build_sequences_grud(df, feat_cols) if is_grud else build_sequences(df, feat_cols)
    pid_arr = np.array([s[2] for s in all_seqs])
    splits = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=42).split(pid_arr))       # v2 outer folds
    for f_, (tr_idx, te_idx) in enumerate(splits):
        done_path = os.path.join(ES_DIR, f"{name}__fold{f_:02d}__histories.json")
        if os.path.exists(done_path): continue
        set_seed(SEED + f_)                                                                     # as in v2 run_cv
        train_raw = [all_seqs[i] for i in tr_idx]; test_raw = [all_seqs[i] for i in te_idx]
        train_s, test_s, _, _ = standardize_fold(train_raw, test_raw, nf)
        perm = np.random.default_rng(SEED + f_).permutation(len(train_raw))                     # v2 inner split
        n_val = max(1, int(round(INNER_VAL_FRAC * len(train_raw))))
        itr_s, ival_s, _, _ = standardize_fold([train_raw[i] for i in perm[n_val:]], [train_raw[i] for i in perm[:n_val]], nf)
        enc_path = os.path.join(ES_DIR, f"{name}__fold{f_:02d}__encoder.pt")
        enc_cls = GRUEncoder if mtype == 'ssl_gru' else GRUDEncoder
        if os.path.exists(enc_path):
            enc = enc_cls(nf, hid=64, drop=0.3, n_layers=2).to(device)
            enc.load_state_dict(torch.load(enc_path, map_location=device))
        else:
            enc = ssl_pretrain_fold(train_s, enc_cls, nf, enc_kwargs={'hid': 64, 'drop': 0.3, 'n_layers': 2},
                                    epochs=SSL_EPOCHS, bs=8, lr=1e-3, wd=1e-4)
            torch.save(enc.state_dict(), enc_path)
        E_tr, E_te, E_itr, E_val = (es_embed(enc, s) for s in (train_s, test_s, itr_s, ival_s))
        fold_hist = {}
        for task in ['classify', 'next']:
            sd = SEED + f_
            # A: v2 protocol (30 epochs, patience 5, refit)
            _, hpA, hA, trA = es_select(E_itr, E_val, task, sd, FT_EPOCHS, PATIENCE, 0.0)
            headA, _ = es_train(E_tr, task, hpA, sd, int(hA['best_epoch']))
            # B: long run, best-validation checkpoint  /  C: long run, refit for B's epoch count
            headB, hpB, hB, trB = es_select(E_itr, E_val, task, sd, ES_MAX_EPOCHS, ES_PATIENCE, ES_MIN_DELTA)
            headC, _ = es_train(E_tr, task, hpB, sd, int(hB['best_epoch']))
            for arm, head in [('v2', headA), ('long_ckpt', headB), ('long_refit', headC)]:
                p, l, ids = es_predict(head, E_te, task)
                tmp = os.path.join(ES_DIR, f"{name}__{task}__fold{f_:02d}__{arm}.csv")
                pd.DataFrame({'participant_id': ids, 'y_true': l.astype(int), 'y_prob': p}).to_csv(tmp + '.tmp', index=False)
                os.replace(tmp + '.tmp', tmp)
            fold_hist[task] = {
                'v2':   {'hp': hpA, 'best_epoch': hA['best_epoch'], 'best_val_auc': hA['best_auc'],
                         'trials': [{'hp': t[1], **{k: t[3][k] for k in ('val_auc', 'val_loss', 'train_loss', 'best_epoch', 'stopped_epoch')}} for t in trA]},
                'long': {'hp': hpB, 'best_epoch': hB['best_epoch'], 'best_val_auc': hB['best_auc'],
                         'trials': [{'hp': t[1], **{k: t[3][k] for k in ('val_auc', 'val_loss', 'train_loss', 'best_epoch', 'stopped_epoch')}} for t in trB]}}
        json.dump(fold_hist, open(done_path + '.tmp', 'w'), default=float); os.replace(done_path + '.tmp', done_path)
        print(f"  {name} fold {f_+1}/{N_FOLDS}  v2 epochs c/n = {fold_hist['classify']['v2']['best_epoch']}/{fold_hist['next']['v2']['best_epoch']}"
              f"  long epochs c/n = {fold_hist['classify']['long']['best_epoch']}/{fold_hist['next']['long']['best_epoch']}"
              f"  {(time.time()-t0)/60:6.1f} min")
print("✅ SSL early-stopping check runs done")
'''

C15 = r'''# ════════════════════════════════════════════════════════
# CELL 15 — SSL EARLY-STOPPING CHECK: DIAGNOSTICS AND TEST RESULTS
# ════════════════════════════════════════════════════════
import glob

# 1) Validation histories: was the 30-epoch cap binding, and did longer training improve validation AUC?
drows = []
for cfg in es_cfgs:
    for path in sorted(glob.glob(os.path.join(ES_DIR, f"{cfg['name']}__fold*__histories.json"))):
        H = json.load(open(path))
        for task, h in H.items():
            for tr in h['long']['trials']:
                va = np.array(tr['val_auc'], float)
                b30 = int(np.nanargmax(va[:FT_EPOCHS])) + 1 if np.any(~np.isnan(va[:FT_EPOCHS])) else np.nan
                drows.append(dict(model=cfg['name'], task=task, lr=tr['hp']['lr'], drop=tr['hp']['drop'],
                                  best_epoch_long=tr['best_epoch'], stopped=tr['stopped_epoch'],
                                  hit_long_cap=tr['stopped_epoch'] >= ES_MAX_EPOCHS,
                                  best_within_30_at_30=(b30 == FT_EPOCHS),
                                  val_auc_best30=np.nanmax(va[:FT_EPOCHS]), val_auc_best_long=np.nanmax(va),
                                  chosen=(tr['hp'] == h['long']['hp'])))
D = pd.DataFrame(drows); D['val_gain_after_30'] = D['val_auc_best_long'] - D['val_auc_best30']
summ = D.groupby(['model', 'task']).agg(
    runs=('stopped', 'size'),
    share_best_at_epoch30=('best_within_30_at_30', 'mean'),
    median_best_epoch_long=('best_epoch_long', 'median'),
    share_hit_long_cap=('hit_long_cap', 'mean'),
    median_val_gain_after_30=('val_gain_after_30', 'median'),
    max_val_gain_after_30=('val_gain_after_30', 'max')).round(3)
pd.set_option('display.width', 250)
print("=== Validation histories (all 4 settings × folds) ===")
print("share_best_at_epoch30: within the first 30 epochs the best validation AUC was at epoch 30 (cap binding in v2)")
print(summ.to_string())

# 2) Test performance of the three arms on the same encoders and folds
def es_load(name, task, arm):
    fs = sorted(glob.glob(os.path.join(ES_DIR, f"{name}__{task}__fold*__{arm}.csv")))
    return [pd.read_csv(f, dtype={'participant_id': str}) for f in fs]

def es_pp(frames):
    d = pd.concat(frames)
    return pd.Series({p: roc_auc_score(g['y_true'], g['y_prob']) for p, g in d.groupby('participant_id') if 0 < g['y_true'].sum() < len(g)})

trows = []
for cfg in es_cfgs:
    for task in ['classify', 'next']:
        F = {arm: es_load(cfg['name'], task, arm) for arm in ['v2', 'long_ckpt', 'long_refit']}
        if not all(len(v) for v in F.values()): continue
        k = min(len(v) for v in F.values())
        fa = {arm: np.array([safe_auc(d['y_true'], d['y_prob']) for d in v[:k]]) for arm, v in F.items()}
        pp = {arm: es_pp(v[:k]) for arm, v in F.items()}
        row = dict(model=cfg['name'], task=task, folds=k)
        for arm in fa: row[f'pooled_{arm}'] = round(np.nanmean(fa[arm]), 4); row[f'pp_median_{arm}'] = round(pp[arm].median(), 3)
        for a, b in [('long_ckpt', 'v2'), ('long_refit', 'v2'), ('long_ckpt', 'long_refit')]:
            d = fa[a] - fa[b]; row[f'pooled_{a}-{b}'] = round(np.nanmean(d), 4)
            row[f'p_{a}-{b}'] = round(stats.wilcoxon(d).pvalue, 4) if k >= 5 and np.any(d != 0) else np.nan
        cm = pp['long_ckpt'].index.intersection(pp['v2'].index); dd = (pp['long_ckpt'][cm] - pp['v2'][cm]).to_numpy()
        row['pp_ckpt-v2_median'] = round(float(np.median(dd)), 3)
        row['pp_ckpt-v2_p'] = round(stats.wilcoxon(dd).pvalue, 4) if np.any(dd != 0) else 1.0
        trows.append(row)
T = pd.DataFrame(trows)
print("\n=== Test performance: v2 protocol vs longer training with early stopping (same encoders, same folds) ===")
print(T.to_string(index=False))
D.to_csv(os.path.join(ES_DIR, 'validation_history_summary.csv'), index=False)
T.to_csv(os.path.join(ES_DIR, 'test_comparison.csv'), index=False)

# 3) Validation curves of the chosen setting, every fold (dashed line = the v2 cap)
fig, axes = plt.subplots(len(es_cfgs), 2, figsize=(11, 3.2 * len(es_cfgs)), squeeze=False)
for r, cfg in enumerate(es_cfgs):
    for c, task in enumerate(['classify', 'next']):
        ax = axes[r][c]
        for path in sorted(glob.glob(os.path.join(ES_DIR, f"{cfg['name']}__fold*__histories.json"))):
            h = json.load(open(path))[task]['long']
            tr = [t for t in h['trials'] if t['hp'] == h['hp']][0]
            ax.plot(range(1, len(tr['val_auc']) + 1), tr['val_auc'], lw=0.9, alpha=0.7)
            ax.plot(tr['best_epoch'], tr['val_auc'][tr['best_epoch'] - 1], 'k.', ms=4)
        ax.axvline(FT_EPOCHS, ls='--', c='grey', lw=1)
        ax.set_title(f"{cfg['name']} — {task}", fontsize=9); ax.set_xlabel('epoch'); ax.set_ylabel('validation AUC')
plt.tight_layout(); plt.savefig(os.path.join(ES_DIR, 'ssl_validation_curves.png'), dpi=200); plt.show()
'''